# Random Forest na Classificação do Desempenho dos Alunos

# Seção 1: Importação do Dataset e Dataset Usado

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_parquet("Dados Limpos")

In [3]:
df.columns

Index(['NU_ANO', 'TP_FAIXA_ETARIA', 'TP_SEXO', 'TP_ESTADO_CIVIL',
       'TP_COR_RACA', 'TP_ST_CONCLUSAO', 'TP_ESCOLA', 'TP_ENSINO',
       'IN_TREINEIRO', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC',
       'TP_SIT_FUNC_ESC', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC',
       'TP_PRESENCA_MT', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC',
       'NU_NOTA_MT', 'TX_RESPOSTAS_CN', 'TX_RESPOSTAS_CH', 'TX_RESPOSTAS_LC',
       'TX_RESPOSTAS_MT', 'TX_GABARITO_CN', 'TX_GABARITO_CH', 'TX_GABARITO_LC',
       'TX_GABARITO_MT', 'TP_STATUS_REDACAO', 'NU_NOTA_COMP1', 'NU_NOTA_COMP2',
       'NU_NOTA_COMP3', 'NU_NOTA_COMP4', 'NU_NOTA_COMP5', 'NU_NOTA_REDACAO',
       'Q001', 'Q002', 'Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009',
       'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018',
       'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025'],
      dtype='object')

# Seção 4: Treinando o Random Forest

In [4]:
def treinar_rf(x_train, y_train, x_test, y_test, n_estimators, max_depth, max_features, min_samples_split, min_samples_leaf):
    
    rf= RandomForestClassifier(
        n_estimators=n_estimators,        
        max_depth=max_depth,            
        max_features = max_features,     
        min_samples_split = min_samples_split,    
        min_samples_leaf = min_samples_leaf,      
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    rf.fit(x_train, y_train)

    y_pred_train = rf.predict(x_train)
    y_pred_test  = rf.predict(x_test)

    ein  = 1 - accuracy_score(y_train, y_pred_train)
    eout = 1 - accuracy_score(y_test,  y_pred_test)

    print(f"\nEin:  {ein:.4f}")
    print(f"Eout: {eout:.4f}")
    print(f"Gap:  {eout - ein:.4f}  {'overfitting' if eout - ein > 0.05 else 'ok'}")
    print("\n" + classification_report(y_test, y_pred_test))

    return rf

In [16]:
df_reduzido = df.sample(n=1_000_000, random_state=42)

In [17]:
def transformar_colunas_ohe(df):
    
    colunas = [
        'Q001','Q002','Q003','Q004','Q006','Q007','Q008','Q009','Q010',
        'Q011','Q012','Q013','Q014','Q015','Q016','Q017','Q018',
        'Q019','Q020','Q021','Q022','Q023','Q024','Q025'
    ]
    
    df = df.dropna(subset=colunas)
    
    df = pd.get_dummies(df, columns=colunas, prefix=colunas, dtype=int)
    
    return df

In [18]:
df_reduzido = transformar_colunas_ohe(df_reduzido)

In [19]:
df_reduzido.columns

Index(['NU_ANO', 'TP_FAIXA_ETARIA', 'TP_SEXO', 'TP_ESTADO_CIVIL',
       'TP_COR_RACA', 'TP_ST_CONCLUSAO', 'TP_ESCOLA', 'TP_ENSINO',
       'IN_TREINEIRO', 'TP_DEPENDENCIA_ADM_ESC',
       ...
       'Q022_E', 'Q023_A', 'Q023_B', 'Q024_A', 'Q024_B', 'Q024_C', 'Q024_D',
       'Q024_E', 'Q025_A', 'Q025_B'],
      dtype='object', length=162)

In [20]:
df_reduzido['MEDIA'] = (df_reduzido['NU_NOTA_CN'] + df_reduzido['NU_NOTA_CH'] + df_reduzido['NU_NOTA_MT']+  df_reduzido['NU_NOTA_LC'] + df_reduzido['NU_NOTA_REDACAO']) / 5

mediana = df_reduzido['MEDIA'].median()
df_reduzido['CLASSE_NOTA'] = (df_reduzido['MEDIA'] < mediana).astype(int)

In [21]:
X = df_reduzido.drop(['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC','TP_SEXO', 'TP_COR_RACA',
       'NU_NOTA_MT', 'TX_RESPOSTAS_CN', 'TX_RESPOSTAS_CH', 'TX_RESPOSTAS_LC',
       'TX_RESPOSTAS_MT', 'TX_GABARITO_CN', 'TX_GABARITO_CH', 'TX_GABARITO_LC',
        'CLASSE_NOTA', 'MEDIA', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT',
       'TX_GABARITO_MT', 'TP_STATUS_REDACAO', 'NU_NOTA_COMP1', 'NU_NOTA_COMP2','IN_TREINEIRO', 
       'NU_NOTA_COMP3', 'NU_NOTA_COMP4', 'NU_NOTA_COMP5', 'NU_NOTA_REDACAO'], axis=1)

y = df_reduzido['CLASSE_NOTA']

In [22]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [24]:
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

rf.fit(x_train, y_train)

print('Ein: %0.4f' % (1 - accuracy_score(y_train, rf.predict(x_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test, rf.predict(x_test))))
print(classification_report(y_test, rf.predict(x_test)))

Ein: 0.0013
Eout: 0.2738
              precision    recall  f1-score   support

           0       0.73      0.69      0.71      9725
           1       0.72      0.76      0.74     10275

    accuracy                           0.73     20000
   macro avg       0.73      0.73      0.73     20000
weighted avg       0.73      0.73      0.73     20000



## Tratando Overfit

In [23]:
n_estimators=60
max_depth=30          
max_features='log2'    
min_samples_split=50
min_samples_leaf=10
    
rf = treinar_rf(x_train, y_train, x_test, y_test, n_estimators, max_depth, max_features, min_samples_split, min_samples_leaf)



Ein:  0.2566
Eout: 0.2746
Gap:  0.0180  ok

              precision    recall  f1-score   support

           0       0.74      0.69      0.71    100005
           1       0.71      0.77      0.74     99995

    accuracy                           0.73    200000
   macro avg       0.73      0.73      0.72    200000
weighted avg       0.73      0.73      0.72    200000



In [24]:
imp = pd.Series(rf.feature_importances_, index=X.columns)
imp = imp.sort_values(ascending=False)

print(imp.head(20))
print(imp.head(20).sum())

TP_DEPENDENCIA_ADM_ESC    0.174785
Q024_A                    0.081394
TP_ESCOLA                 0.065279
Q003_D                    0.042231
Q006_B                    0.036286
Q018_B                    0.033469
Q004_D                    0.028881
Q010_A                    0.022033
Q008_B                    0.020498
Q025_B                    0.020269
Q018_A                    0.019094
TP_FAIXA_ETARIA           0.018962
Q016_B                    0.018420
Q013_A                    0.018065
Q014_A                    0.017825
Q024_B                    0.017803
Q003_E                    0.016704
NU_ANO                    0.015540
Q002_G                    0.014181
Q004_E                    0.012414
dtype: float64
0.6941336429091429
